# 📝 Week 17 — Advanced Prompt Engineering
## Microlearning Content for Internal Newsletter

**Scenario:** You work on the Learning & Communications Team at a global tech company. Your task is to use generative AI to transform a dense internal policy paragraph into clear, employee-facing microlearning content.

---

### 📄 Original Policy Text (Source Input)

> *"Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications."*

---

## ⚙️ Setup — Install & Configure

Run this cell first to install the OpenAI library and configure your API key.

In [ ]:
# Install required libraries (no API key needed)
!pip install transformers torch -q

from transformers import pipeline
import textwrap

# Load a free lightweight model — works on CPU in Colab, no API key needed
print("⏳ Loading model... (may take ~1 minute on first run)")
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    device_map="auto"
)
print("✅ Model loaded!")

def call_model(prompt, max_new_tokens=200):
    """Send a prompt to SmolLM2 and return only the new generated text."""
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_new_tokens, do_sample=False)
    # Extract only the assistant reply
    return output[0]["generated_text"][-1]["content"].strip()

# Source text used across all steps
SOURCE_TEXT = (
    "Employees must ensure that all remote access to internal systems is established "
    "via the approved secure VPN. Under no circumstances should unsecured connections "
    "or personal devices lacking endpoint protection be used to access proprietary "
    "data or sensitive communications."
)

print("\n✅ Setup complete. No API key needed.")


---
## Step 1 — Craft a Prompt: Tone, Format & Length Control

**Goal:** Rewrite the policy text in a friendly tone, using bullet points, paraphrasing the original (no direct quotes), and staying under 75 words.

**Technique used:** Role prompting + instruction-based prompting + output constraints

In [ ]:
# ── Step 1: Prompt Design ──────────────────────────────────────────────────

prompt_step1 = (
    "You are an internal communications specialist writing for a company newsletter.\n"
    "Your audience is non-technical employees.\n\n"
    "Rewrite the following policy excerpt according to these rules:\n"
    "- Use a friendly, encouraging tone\n"
    "- Organize the information into bullet points\n"
    "- Paraphrase the content — do NOT copy or quote any part of the original text\n"
    "- Keep the total response strictly under 75 words\n\n"
    "Policy excerpt:\n"
    + SOURCE_TEXT
)

print("📋 PROMPT SENT TO MODEL:")
print("-" * 60)
print(prompt_step1)
print("-" * 60)


In [ ]:
# ── Run Step 1 ─────────────────────────────────────────────────────────────

output_step1 = call_model(prompt_step1)
word_count_1 = len(output_step1.split())

print("🤖 MODEL OUTPUT:")
print("-" * 60)
print(output_step1)
print("-" * 60)
print(f"📊 Word count: {word_count_1} words")

---
## Step 2 — Evaluate the Output

Review the model's output against the five quality criteria. Fill in your observations below.

In [ ]:
# ── Step 2: Manual Evaluation ──────────────────────────────────────────────
# Fill in your observations for each criterion

evaluation = {
    "Relevance": {
        "question": "Does the output stick to the message of the source content?",
        "observation": "✏️ YOUR OBSERVATION HERE"   # <-- edit this
    },
    "Clarity": {
        "question": "Is the message easier to understand for non-technical staff?",
        "observation": "✏️ YOUR OBSERVATION HERE"
    },
    "Structure": {
        "question": "Is the information formatted into clear bullet points?",
        "observation": "✏️ YOUR OBSERVATION HERE"
    },
    "Tone": {
        "question": "Is the language appropriate for internal employee communication?",
        "observation": "✏️ YOUR OBSERVATION HERE"
    },
    "Length": {
        "question": "Does it stay under 75 words?",
        "observation": f"Word count detected: {len(output_step1.split())} — {'✅ PASS' if len(output_step1.split()) < 75 else '❌ FAIL — exceeds 75 words'}"
    },
    "Factual Accuracy": {
        "question": "Did the model make up or remove any critical security details?",
        "observation": "✏️ YOUR OBSERVATION HERE"
    }
}

print("📊 EVALUATION SCORECARD")
print("=" * 60)
for criterion, data in evaluation.items():
    print(f"\n🔹 {criterion}")
    print(f"   Q: {data['question']}")
    print(f"   A: {data['observation']}")
print("=" * 60)

---
## Step 3 — Detect & Mitigate Hallucinations

**Goal:** Strengthen the prompt to prevent the model from adding content not present in the source (e.g., new policy terms, tool names, extra recommendations).

**Technique:** Adding a strict scope constraint — *"Only paraphrase the content in the input text."*

In [ ]:
# ── Step 3: Anti-Hallucination Prompt ─────────────────────────────────────

prompt_step3 = (
    "You are an internal communications specialist writing for a company newsletter.\n"
    "Your audience is non-technical employees.\n\n"
    "Rewrite the following policy excerpt according to these rules:\n"
    "- Use a friendly, encouraging tone\n"
    "- Organize the information into bullet points\n"
    "- Paraphrase the content — do NOT copy or quote any part of the original text\n"
    "- Keep the total response strictly under 75 words\n"
    "- IMPORTANT: Only paraphrase the content in the input text.\n"
    "  Do NOT add new recommendations, tools, technologies, or policy terms\n"
    "  that are not explicitly mentioned in the excerpt below.\n\n"
    "Policy excerpt:\n"
    + SOURCE_TEXT
)

output_step3 = call_model(prompt_step3)
word_count_3 = len(output_step3.split())

print("🛡️ ANTI-HALLUCINATION OUTPUT:")
print("-" * 60)
print(output_step3)
print("-" * 60)
print(f"📊 Word count: {word_count_3} words")

print("\n📝 KEY CHANGES IN THIS PROMPT:")
print("  ✅ Added explicit constraint: 'Only paraphrase content in the input text'")
print("  ✅ Explicitly forbids adding new technologies, tools, or policy terms")
print("  ✅ Reduces model's tendency to embellish with generic security advice")


---
## Step 4 — Paraphrasing Deep Dive (Junior Intern Audience)

**Goal:** Rewrite the policy for a junior intern — plain language, short phrases, max 4 bullet points, no corporate/legal jargon, supportive tone.

**Technique:** Audience targeting + strict output constraints

In [ ]:
# ── Step 4: Intern-Audience Paraphrase ────────────────────────────────────

prompt_step4 = (
    "You are writing a quick tip for a junior intern who just joined the company.\n"
    "They are not familiar with corporate or legal language.\n\n"
    "Rewrite the following policy excerpt so that:\n"
    "- The language is plain and simple — like explaining to a friend\n"
    "- You use short, punchy phrases\n"
    "- You use exactly 4 bullet points (no more, no less)\n"
    "- There is zero corporate or legal jargon\n"
    "- The tone is supportive and informative, not scary or authoritative\n"
    "- Only paraphrase what is in the input — do not add new content\n\n"
    "Policy excerpt:\n"
    + SOURCE_TEXT
)

output_step4 = call_model(prompt_step4)

print("🎓 INTERN-FRIENDLY OUTPUT:")
print("-" * 60)
print(output_step4)
print("-" * 60)
bullet_count = output_step4.count("\n-") + output_step4.count("\n•") + output_step4.count("\n*")
print(f"📊 Estimated bullet points: ~{bullet_count}")


---
## Step 5 — Quote Extraction Variant

**Goal:** Extract the single best direct quote from the original that captures the core security policy message — and reflect on *when* quoting is appropriate vs. risky.

In [ ]:
# ── Step 5: Quote Extraction ───────────────────────────────────────────────

prompt_step5 = (
    "You are reviewing an internal security policy.\n\n"
    "From the following excerpt, identify and extract the ONE direct quote (verbatim)\n"
    "that best captures the core security requirement.\n\n"
    "Return ONLY:\n"
    "1. The exact quote in quotation marks\n"
    "2. One sentence explaining why you chose this quote\n\n"
    "Do not rephrase or paraphrase — copy the text exactly as written.\n\n"
    "Policy excerpt:\n"
    + SOURCE_TEXT
)

output_step5 = call_model(prompt_step5)

print("💬 EXTRACTED QUOTE:")
print("-" * 60)
print(output_step5)
print("-" * 60)


In [ ]:
# ── Step 5: Reflection Questions ──────────────────────────────────────────

reflection = """
📌 REFLECTION: Quoting vs. Paraphrasing in Internal Communications
=================================================================

❓ Q1: In what kind of internal communication would QUOTING be more appropriate?

   ✅ Answer:
   Quoting is most appropriate when:
   - Legal or compliance accuracy is critical (e.g., audit reports, policy notices,
     disciplinary communications) — the exact wording carries legal weight.
   - You are referencing a rule that employees may be held accountable for,
     and misrepresenting its wording could create liability.
   - You are citing leadership statements, official announcements, or regulatory
     language where precision matters more than readability.

❓ Q2: When might QUOTING pose a risk?

   ⚠️ Answer:
   Quoting can be risky when:
   - The original text is overly technical or full of jargon — a direct quote
     may confuse non-technical readers and cause non-compliance due to
     misunderstanding, not malicious intent.
   - The policy language is dense or legalistic, leading employees to disengage.
   - The quote is taken out of context, distorting its intended meaning.
   - In microlearning contexts, long verbatim quotes undermine the goal of
     simplification and accessibility.
"""

print(reflection)

---
## 🧩 Summary — Full Prompt Development Cycle

| Step | Task | Technique |
|------|------|-----------|
| 1 | Rewrite with tone/format/length control | Role prompting + output constraints |
| 2 | Evaluate output quality | Manual rubric review |
| 3 | Mitigate hallucinations | Scope-limiting instruction |
| 4 | Paraphrase for intern audience | Audience targeting + jargon exclusion |
| 5 | Extract key quote + reflection | Instruction-based + critical analysis |

> **Key takeaway:** A good prompt is not written once — it is designed, tested, evaluated, and refined iteratively. Small changes in wording (adding a scope constraint, specifying audience, limiting bullet count) can significantly change output quality.

In [ ]:
# ── Final Comparison: All Outputs Side by Side ────────────────────────────

print("=" * 60)
print("📋 FINAL COMPARISON — ALL OUTPUTS")
print("=" * 60)

outputs = {
    "Step 1 — Tone/Format/Length Control": output_step1,
    "Step 3 — Anti-Hallucination Version": output_step3,
    "Step 4 — Intern Audience Version": output_step4,
    "Step 5 — Quote Extraction": output_step5,
}

for title, output in outputs.items():
    print(f"\n🔹 {title}")
    print("-" * 50)
    print(textwrap.fill(output, width=70))
    print(f"   ({len(output.split())} words)")

print("\n" + "=" * 60)
print("✅ Assignment complete!")